In [ ]:
# 🎧 Detección de la Mejor Toma Musical (LOCAL) con YAMNet
# - Métricas separadas (Sonido/Ejecución)
# - Normalización por carpeta (min–max)
# - Logs en consola
# - CSV con crudos y normalizados

import os
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from pydub import AudioSegment
import tensorflow as tf
import tensorflow_hub as hub

# ========== Config ==========
ENSAYO_ROOT = Path("/Users/javiercordero/workspace/jac/zapa-ia")  # 👈 tu raíz local
TOP_N_POR_CARPETA = 800       # poné 5 si querés solo top5
PESO_SONIDO = 0.4
PESO_EJEC   = 0.6
CSV_SALIDA  = "ranking_topN_tomas_por_carpeta.csv"
DECIMAL_COMA = True          # True si querés coma decimal para Sheets AR
SR = 16000
# ===========================

# Modelo YAMNet
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# -------- Audio --------
def load_mp3(file_path: str, sr: int = SR) -> np.ndarray:
    audio = AudioSegment.from_mp3(file_path)
    audio = audio.set_channels(1).set_frame_rate(sr)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32) / 32768.0
    return samples

# -------- Score: SONIDO --------
def sound_quality_score(waveform: np.ndarray, sr: int = SR) -> float:
    try:
        tempo, beats = librosa.beat.beat_track(y=waveform, sr=sr)
        beat_var = np.std(np.diff(beats)) if len(beats) > 2 else 100.0

        rms = librosa.feature.rms(y=waveform)[0]
        avg_rms = float(np.mean(rms))

        centroid = librosa.feature.spectral_centroid(y=waveform, sr=sr)[0]
        avg_centroid = float(np.mean(centroid))

        intervals = librosa.effects.split(waveform, top_db=30)
        silence_ratio = 1 - sum(i[1]-i[0] for i in intervals) / len(waveform)

        waveform_tf = tf.convert_to_tensor(waveform, dtype=tf.float32)
        scores, embeddings, spectrogram = yamnet_model(waveform_tf)
        mean_scores = tf.reduce_mean(scores, axis=0).numpy()

        class_map_path = tf.keras.utils.get_file(
            'yamnet_class_map.csv',
            'https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/yamnet_class_map.csv'
        )
        df_labels = pd.read_csv(class_map_path)
        penalty_labels = {"Hum", "Buzz", "Static", "Noise"}
        penalty_score = sum(
            mean_scores[i] for i, label in enumerate(df_labels.display_name) if label in penalty_labels
        )

        score = (avg_rms * 0.4) - (beat_var * 0.2) + (avg_centroid * 0.2) - (silence_ratio * 0.2) - (penalty_score * 5)
        return float(score)
    except Exception as e:
        print(f"⚠️ sound_quality_score error: {e}", flush=True)
        return 0.0

# -------- Score: EJECUCIÓN (tightness) --------
def tightness_score(y: np.ndarray, sr: int = SR) -> float:
    try:
        tempo, beats = librosa.beat.beat_track(y=y, sr=sr, units='time')
        if len(beats) < 4:
            return 0.0
        ibis = np.diff(beats)
        ibi_std = float(np.std(ibis))                          # menor = mejor
        tempo_drift = float(np.std(librosa.util.normalize(ibis)))

        onsets = librosa.onset.onset_detect(y=y, sr=sr, units='time', backtrack=True)
        idx = np.searchsorted(beats, onsets) - 1
        valid = (idx >= 0) & (idx < len(beats) - 1)
        group = {}
        for t, b in zip(onsets[valid], idx[valid]):
            group.setdefault(b, []).append(t - beats[b])
        spreads = [np.std(v) for v in group.values() if len(v) >= 2]
        intra_spread = float(np.median(spreads)) if spreads else 0.05  # menor = mejor

        y_harm, y_perc = librosa.effects.hpss(y)
        env_bass = librosa.onset.onset_strength(y=librosa.effects.preemphasis(y_harm), sr=sr)
        env_perc = librosa.onset.onset_strength(y=y_perc, sr=sr)
        L = min(len(env_bass), len(env_perc))
        if L < 10:
            return 0.0
        corr = np.correlate(librosa.util.normalize(env_bass[:L]),
                            librosa.util.normalize(env_perc[:L]),
                            mode='valid')
        perc_bass_sync = float(np.max(corr)) if len(corr) else 0.0  # mayor = mejor

        s_ibi    = np.exp(-5 * ibi_std)
        s_drift  = np.exp(-5 * tempo_drift)
        s_spread = np.exp(-20 * intra_spread)
        s_sync   = float(np.clip(perc_bass_sync, 0, 1))

        tight = 0.35*s_ibi + 0.25*s_drift + 0.25*s_spread + 0.15*s_sync
        return float(100 * tight)
    except Exception as e:
        print(f"⚠️ tightness_score error: {e}", flush=True)
        return 0.0

def minmax_norm(series: pd.Series) -> pd.Series:
    s = series.astype(float)
    s_min, s_max = float(np.nanmin(s)), float(np.nanmax(s))
    if not np.isfinite(s_min) or not np.isfinite(s_max) or s_max == s_min:
        return pd.Series([0.5]*len(s), index=s.index)
    return (s - s_min) / (s_max - s_min)

# ---------- Pipeline ----------
resultados = []

for carpeta in sorted(Path(ENSAYO_ROOT).iterdir()):
    if not carpeta.is_dir():
        continue
    mp3s = sorted(carpeta.glob("*.mp3"))
    if not mp3s:
        continue

    print(f"\n📁 Carpeta: {carpeta.name}  (archivos: {len(mp3s)})", flush=True)
    tomas = []

    for mp3_file in mp3s:
        print(f"Procesando: {mp3_file.name}", flush=True)
        try:
            y = load_mp3(str(mp3_file), sr=SR)
            s_son = sound_quality_score(y, sr=SR)
            s_eje = tightness_score(y, sr=SR)
            print(f"  ↳ Sonido_raw={s_son:.4f} | Ejec_raw={s_eje:.4f}", flush=True)

            tomas.append({
                "Ensayo": carpeta.name,
                "Toma": mp3_file.name,
                "Score_Sonido_raw": s_son,
                "Score_Ejecucion_raw": s_eje,
                "Ruta": str(mp3_file.resolve())
            })
        except Exception as e:
            print(f"❌ Error procesando {mp3_file.name}: {e}", flush=True)

    if not tomas:
        continue

    df_carp = pd.DataFrame(tomas)
    # Normalización por carpeta
    df_carp["Score_Sonido_norm"]    = minmax_norm(df_carp["Score_Sonido_raw"])
    df_carp["Score_Ejecucion_norm"] = minmax_norm(df_carp["Score_Ejecucion_raw"])

    df_carp["Score_Final"] = (
        PESO_SONIDO * df_carp["Score_Sonido_norm"] +
        PESO_EJEC   * df_carp["Score_Ejecucion_norm"]
    )

    # Top por carpeta
    df_top = df_carp.sort_values("Score_Final", ascending=False).head(TOP_N_POR_CARPETA)
    print("\n🏁 Top carpeta (muestra hasta 10):", flush=True)
    print(df_top.head(10)[["Toma","Score_Final","Score_Sonido_raw","Score_Ejecucion_raw"]]
          .to_string(index=False, float_format=lambda x: f"{x:.4f}"),
          flush=True)

    resultados.extend(df_top.to_dict(orient="records"))

# ---------- Export ----------
if resultados:
    df = pd.DataFrame(resultados)
    df = df.sort_values(by=["Ensayo", "Score_Final"], ascending=[True, False]).reset_index(drop=True)

    # Mostrar primeras 50 en consola
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', 200)
    print("\n📊 Resultados globales (primeras 50):")
    print(df.head(50).to_string(index=False, float_format=lambda x: f"{x:.4f}"), flush=True)

    # CSV
    if DECIMAL_COMA:
        df.to_csv(CSV_SALIDA, index=False, decimal=',')
    else:
        df.to_csv(CSV_SALIDA, index=False)

    print(f"\n✅ Archivo exportado como '{CSV_SALIDA}'")
else:
    print("No se encontraron tomas para procesar.")



📁 Carpeta: nebulosa  (archivos: 850)
Procesando: 45 rpm .mp3
  ↳ Sonido_raw=421.1678 | Ejec_raw=71.3470
Procesando: 990.mp3
  ↳ Sonido_raw=175.1394 | Ejec_raw=69.5011
Procesando: BREATHE.mp3
  ↳ Sonido_raw=338.8003 | Ejec_raw=69.5940
Procesando: Bossa (1).mp3
  ↳ Sonido_raw=138.3269 | Ejec_raw=94.4939
Procesando: Bossa.mp3
  ↳ Sonido_raw=138.3269 | Ejec_raw=94.4939
Procesando: Copia de el playadito.mp3
  ↳ Sonido_raw=343.0911 | Ejec_raw=70.2235
Procesando: Copia de pobre mate multipista.mp3
  ↳ Sonido_raw=339.6062 | Ejec_raw=71.0003
Procesando: Copy of cabras 2da parte.mp3
